<a href="https://colab.research.google.com/github/peperjet/bc-ml/blob/main/flight_delay/flight_delay_260401.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

항공편 지연 예측 AI

In [56]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

warnings.filterwarnings('ignore')

In [57]:
# 시드 고정

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(42)

데이터 불러오기


In [58]:
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')
sample_submission = pd.read_csv('/content/sample_submission.csv')

print(train.shape, test.shape, sample_submission.shape)
train.head()

(1000000, 19) (1000000, 18) (1000000, 3)


,ID,Month,Day_of_Month,Estimated_Departure_Time,Estimated_Arrival_Time,Cancelled,Diverted,Origin_Airport,Origin_Airport_ID,Origin_State,Destination_Airport,Destination_Airport_ID,Destination_State,Distance,Airline,Carrier_Code(IATA),Carrier_ID(DOT),Tail_Number,Delay
0,TRAIN_000000,4,15,NaN,NaN,0,0,OKC,13851,Oklahoma,HOU,12191,Texas,419.0,Southwest Airlines Co.,WN,19393.0,N7858A,NaN
1,TRAIN_000001,8,15,740.0,1024.0,0,0,ORD,13930,Illinois,SLC,14869,Utah,1250.0,SkyWest Airlines Inc.,UA,20304.0,N125SY,NaN
2,TRAIN_000002,9,6,1610.0,1805.0,0,0,CLT,11057,North Carolina,LGA,12953,New York,544.0,American Airlines Inc.,AA,19805.0,N103US,NaN
3,TRAIN_000003,7,10,905.0,1735.0,0,0,LAX,12892,California,EWR,11618,New Jersey,2454.0,United Air Lines Inc.,UA,NaN,N595UA,NaN
4,TRAIN_000004,1,11,900.0,1019.0,0,0,SFO,14771,California,ACV,10157,California,250.0,SkyWest Airlines Inc.,UA,20304.0,N161SY,NaN


In [59]:
# 타겟  상태 확인

print(train['Delay'].value_counts(dropna=False))

Delay
NaN            744999
Not_Delayed    210001
Delayed         45000
Name: count, dtype: int64


*   NaN 744,999개 → 테스트용처럼 정답 없는 행
*   Not_Delayed 210,001개
*   Delayed 45,000개



In [61]:
# 1. 라벨 있는 데이터만 분리
train_labeled = train[train['Delay'].notnull()].copy()

print(train_labeled['Delay'].value_counts())

Delay
Not_Delayed    210001
Delayed         45000
Name: count, dtype: int64


In [62]:
# 인코딩

target_le = LabelEncoder()
train_labeled['Delay'] = target_le.fit_transform(train_labeled['Delay'])

print(target_le.classes_)
print(train_labeled['Delay'].value_counts())

['Delayed' 'Not_Delayed']
Delay
1    210001
0     45000
Name: count, dtype: int64


In [63]:
# 범주형 인코딩
cat_cols = [
    'Origin_Airport',
    'Origin_State',
    'Destination_Airport',
    'Destination_State',
    'Airline',
    'Carrier_Code(IATA)',
    'Tail_Number'
]

for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train_labeled[col], test[col]], axis=0).astype(str))

    train_labeled[col] = le.transform(train_labeled[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

print('범주형 인코딩 완료')

범주형 인코딩 완료


In [66]:
# 학습용 데이터 만들기
X = train_labeled.drop(columns=['ID', 'Delay'])
y = train_labeled['Delay']
X_test = test.drop(columns=['ID'])

print(X.shape, y.shape, X_test.shape)
print(X.isnull().sum().sum(), X_test.isnull().sum().sum())

(255001, 17) (255001,) (1000000, 17)
83295 327038


In [67]:
print(X.isnull().sum()[X.isnull().sum() > 0])
print(X_test.isnull().sum()[X_test.isnull().sum() > 0])

Estimated_Departure_Time    27841
Estimated_Arrival_Time      27684
Carrier_ID(DOT)             27770
dtype: int64
Estimated_Departure_Time    108984
Estimated_Arrival_Time      109048
Carrier_ID(DOT)             109006
dtype: int64


In [68]:
for col in X.columns:
    if X[col].isnull().sum() > 0:
        fill_value = X[col].median()
        X[col] = X[col].fillna(fill_value)
        X_test[col] = X_test[col].fillna(fill_value)

In [69]:
print(X.isnull().sum().sum(), X_test.isnull().sum().sum())

0 0


In [70]:
# CV + LightGBM
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

oof_pred = np.zeros((len(X), len(target_le.classes_)))
test_pred = np.zeros((len(X_test), len(target_le.classes_)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=len(target_le.classes_),
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    )

    model.fit(X_train, y_train)

    val_pred = model.predict_proba(X_val)
    oof_pred[val_idx] = val_pred
    test_pred += model.predict_proba(X_test) / skf.n_splits

    fold_score = log_loss(y_val, val_pred)
    print(f'Fold {fold+1} LogLoss: {fold_score:.5f}')

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.049888 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2027
[LightGBM] [Info] Number of data points in the train set: 170000, number of used features: 15
[LightGBM] [Info] Start training from score -1.734601
[LightGBM] [Info] Start training from score -0.194156
Fold 1 LogLoss: 0.44171
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027837 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2023
[LightGBM] [Info] Number of data points in the train set: 170001, number of used features: 15
[LightGBM] [Info] Start training from score -1.734607
[LightGBM] [Info] Start training from score -0.194155
Fold 2 LogLoss: 0.44209
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007611 seconds.
You can set `force_row_wise=true` to remove the over

In [71]:
cv_score = log_loss(y, oof_pred)
print('CV LogLoss:', cv_score)

CV LogLoss: 0.4420259551813685


In [79]:
submission = pd.DataFrame({
    'ID': sample_submission['ID'],
    'Not_Delayed': test_pred[:, list(target_le.classes_).index('Not_Delayed')],
    'Delayed': test_pred[:, list(target_le.classes_).index('Delayed')]
})

submission.to_csv('/content/lgb_baseline_submission.csv', index=False)
submission.head()

,ID,Not_Delayed,Delayed
0,TEST_000000,0.834382,0.165618
1,TEST_000001,0.806440,0.193560
2,TEST_000002,0.713445,0.286555
3,TEST_000003,0.701663,0.298337
4,TEST_000004,0.623623,0.376377


In [78]:
submission.head()
submission.shape

(1000000, 3)

In [80]:
from google.colab import files
files.download('/content/lgb_baseline_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 시간 관련 피처 생성
`Estimated_Departure_Time`과 `Estimated_Arrival_Time`에서 시간과 분을 추출하고, 예상 비행 시간(`Flight_Duration`)을 계산합니다.

In [ ]:
# Helper function to extract time features
def extract_time_features(df, time_col):
    # Convert float times (e.g., 740.0, 1610.0) to proper time format for calculations
    # Handle cases where time is 2400 (midnight) by converting to 0
    df[time_col] = df[time_col].replace(2400, 0)

    # Extract hour and minute
    df[f'{time_col}_Hour'] = (df[time_col] // 100).astype(int)
    df[f'{time_col}_Minute'] = (df[time_col] % 100).astype(int)

    return df

# Apply to Estimated_Departure_Time
X = extract_time_features(X.copy(), 'Estimated_Departure_Time')
X_test = extract_time_features(X_test.copy(), 'Estimated_Departure_Time')

# Apply to Estimated_Arrival_Time
X = extract_time_features(X.copy(), 'Estimated_Arrival_Time')
X_test = extract_time_features(X_test.copy(), 'Estimated_Arrival_Time')

# Calculate Flight_Duration in minutes
X['Flight_Duration'] = (X['Estimated_Arrival_Time_Hour'] * 60 + X['Estimated_Arrival_Time_Minute']) - \
                        (X['Estimated_Departure_Time_Hour'] * 60 + X['Estimated_Departure_Time_Minute'])
X_test['Flight_Duration'] = (X_test['Estimated_Arrival_Time_Hour'] * 60 + X_test['Estimated_Arrival_Time_Minute']) - \
                             (X_test['Estimated_Departure_Time_Hour'] * 60 + X_test['Estimated_Departure_Time_Minute'])

# Handle overnight flights where arrival time is earlier than departure time (e.g., 2300 dep, 0100 arr)
# Add 24 hours (1440 minutes) to duration if it's negative
X.loc[X['Flight_Duration'] < 0, 'Flight_Duration'] += 1440
X_test.loc[X_test['Flight_Duration'] < 0, 'Flight_Duration'] += 1440

# Drop original time columns
X = X.drop(columns=['Estimated_Departure_Time', 'Estimated_Arrival_Time'])
X_test = X_test.drop(columns=['Estimated_Departure_Time', 'Estimated_Arrival_Time'])

print("Features after time engineering for X:")
display(X.head())
print("\nFeatures after time engineering for X_test:")
display(X_test.head())